In [1]:
import os
from mistralai import Mistral
import os
import json
import random
import time

API Key (Free model)

In [2]:
api_key = "ujI60UR6Fe5jel48SAtfnMiN5Skxfwhq"
model =  "open-mistral-nemo"
client = Mistral(api_key=api_key)

Load data

In [3]:
import pandas as pd
from pathlib import Path
# Use a relative path so it works on any machine
csv_path = Path("Data/clean/filtered_fabi.csv")
df = pd.read_csv(csv_path)

print(f"✅ Loaded {len(df)} documents.")
# Print the first few rows of the DataFrame 
print(df.columns.tolist())


FileNotFoundError: [Errno 2] No such file or directory: 'Data\\clean\\filtered_fabi.csv'

Extract needed text from document

In [ ]:
def get_full_text(row):
    titel = row.get("titel", "")
    leitsatz = row.get("leitsatz", "")
    gruende = row.get("gruende", "")
    return f"Titel: {titel}\n\nLeitsatz: {leitsatz}\n\nGründe:\n{gruende}"

# Pick one random document to start
sample_row = df.sample(1).iloc[0]
doc_text = get_full_text(sample_row)

print(doc_text[:500])  # Preview



Titel: Anspruch gegen die Versorgungsanstalt des Bundes und der Länder (VBL) auf Zuteilung und Gutschrift von Bonuspunkten i.S.e. Überschussbeteiligung; Entscheidungsbefugnis des Verwaltungsrats der VBL über die Gewährung von Überschussbeteiligungen dem Grunde nach und gegebenenfalls über die Höhe der Beteiligungen; Richterliche Kontrolle der als Satzungsbestimmungen verfassten Allgemeinen Versicherungsbedingungen der Versicherungsanstalt; Unangemessene Benachteiligung des Einzelnen durch die Ve


Extract ontology components

In [ ]:
prompt = f"""
Du bist ein Ontologie-Experte für deutsches Zivilrecht. Analysiere den folgenden gerichtlichen Textauszug und extrahiere:

1. OWL-Klassen (z. B. Vertrag, Person, Pflichtverletzung)
2. Objekt-Eigenschaften (z. B. betrifft, führt_zu, hat_Partei)
3. Daten-Eigenschaften (z. B. hatDatum, hatBegründung)
4. Beispiel-Individuen (optional)

Text:
---------
{doc_text}
---------
Gib das Ergebnis in strukturierter Liste auf Deutsch aus.
"""


Call client

In [ ]:
response = client.chat.complete(
    model=model,
    messages=[{"role": "user", "content": prompt}]
)

ontology_output = response.choices[0].message.content
print(ontology_output)


1. OWL-Klassen:
* Vertrag (z.B. Versicherungsvertrag)
* Person (z.B. Versicherungsnehmer, Verwaltungsrat der VBL)
* Pflichtverletzung (z.B. Versagung der Überschussbeteiligung)
* Überschussbeteiligung
* Versicherungsanstalt (z.B. VBL)
* Verwaltungsrat
* Allgemeine Versicherungsbedingungen
* Anspruch (z.B. auf Zuteilung und Gutschrift von Bonuspunkten, auf Auskunft über erzielte Überschüsse)
1. Objekt-Eigenschaften:
* betrifft (z.B. der Verwaltungsrat der VBL betrifft die Gewährung von Überschussbeteiligungen)
* hat_Partei (z.B. der Versicherungsvertrag hat den Versicherungsnehmer und die VBL als Parteien)
* führt_zu (z.B. die Versagung der Überschussbeteiligung führt zu einer unangemessenen Benachteiligung des Einzelnen)
1. Daten-Eigenschaften:
* hatDatum (z.B. der Vertrag hat ein Abschlussdatum)
* hatBegründung (z.B. die Entscheidung des Verwaltungsrats hat eine Begründung)
1. Beispiel-Individuen (optional):
* Ein konkreter Versicherungsvertrag zwischen einem Versicherungsnehmer und d